# Retail — Spark gold model

Builds `retail.silver.orders` and the gold sales tables from the Spark-generated bronze orders.
Runs as `retail.analyst`; all writes are governed by the `retail` catalog grants.


In [ ]:
import stardelt
spark = stardelt.spark()
print("spark", spark.version)


In [ ]:
silver = spark.sql('''
SELECT o.order_id, o.order_ts, o.order_date, o.order_month,
       o.customer_id, o.product_id, o.store_id, o.quantity,
       ROUND(o.quantity * o.unit_price * (1 - o.discount), 2) AS revenue,
       o.channel, o.status,
       p.category, p.product_name,
       s.city, s.country, s.region, s.store_name
FROM retail.bronze.orders o
JOIN retailref.reference.products p ON o.product_id = p.product_id
JOIN retailref.reference.stores s ON o.store_id = s.store_id
WHERE o.status = 'completed' AND o.quantity > 0 AND o.unit_price > 0
''')
silver.writeTo("retail.silver.orders").using("iceberg").partitionedBy("order_month").createOrReplace()
print("silver", spark.table("retail.silver.orders").count())


In [ ]:
daily = spark.sql('''
SELECT order_date, country,
       count(*) AS orders,
       sum(quantity) AS units,
       ROUND(sum(revenue), 2) AS revenue,
       ROUND(avg(revenue), 2) AS avg_order_value
FROM retail.silver.orders
GROUP BY order_date, country
''')
daily.writeTo("retail.gold.daily_sales").createOrReplace()
print("daily", daily.count())


In [ ]:
category = spark.sql('''
SELECT category,
       count(*) AS orders,
       sum(quantity) AS units,
       ROUND(sum(revenue), 2) AS revenue,
       ROUND(avg(revenue), 2) AS avg_order_value
FROM retail.silver.orders
GROUP BY category
''')
category.writeTo("retail.gold.category_sales").createOrReplace()
print("category", category.count())


In [ ]:
stores = spark.sql('''
SELECT store_id, store_name, city, country,
       count(*) AS orders,
       ROUND(sum(revenue), 2) AS revenue,
       ROUND(avg(revenue), 2) AS avg_order_value
FROM retail.silver.orders
GROUP BY store_id, store_name, city, country
''')
stores.writeTo("retail.gold.store_sales").createOrReplace()
print("stores", stores.count())


In [ ]:
kpis = spark.sql('''
SELECT count(*) AS orders,
       count(DISTINCT customer_id) AS customers,
       count(DISTINCT product_id) AS products,
       count(DISTINCT store_id) AS stores,
       ROUND(sum(revenue), 2) AS revenue,
       ROUND(avg(revenue), 2) AS avg_order_value,
       ROUND(sum(quantity), 0) AS units,
       min(order_date) AS first_day,
       max(order_date) AS last_day
FROM retail.silver.orders
''')
kpis.writeTo("retail.gold.sales_kpis").createOrReplace()
kpis.show(truncate=False)


In [ ]:
spark.sql("SELECT * FROM retail.gold.sales_kpis").show(truncate=False)
spark.sql("SELECT category, revenue FROM retail.gold.category_sales ORDER BY revenue DESC").show(truncate=False)
